# seleccion_modelo

Proyecto ARIMA / SARIMA / ARIMAX / SARIMAX
Modelación epidemiológica con variables meteorológicas.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PowerTransformer, StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# 1. CARGA DE BASE DE DATOS ORIGINAL
# ============================================================
ruta_excel = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\1_datos_semana_epi_meteo\datos_semanal_meteo_epi.xlsx"
df_orig = pd.read_excel(ruta_excel)
df_orig['fecha'] = pd.to_datetime(df_orig['fecha'])
df_orig.set_index('fecha', inplace=True)

df_transf = df_orig[['año', 'semana_epi']].copy()

In [4]:
# ============================================================
# 2. APLICACIÓN DE LAS TRANSFORMACIONES SOLICITADAS
# ============================================================
df_transf['casos_ln'] = np.log1p(df_orig['casos_dengue'])
df_transf['prec_ln'] = np.log1p(df_orig['prec'])
df_transf['dias_lluvia_ln'] = np.log1p(df_orig['dias_lluvia'])

pt_boxcox = PowerTransformer(method='box-cox')
vars_local = ['temp_max', 'hum_esp', 'vel_vi_max']
for var in vars_local:
    datos_var = df_orig[[var]].copy()
    if (datos_var <= 0).any().any():
        datos_var = datos_var - datos_var.min() + 0.01
    df_transf[f'{var}_bc'] = pt_boxcox.fit_transform(datos_var)

pt_yeo = PowerTransformer(method='yeo-johnson')
df_transf['sst_yj'] = pt_yeo.fit_transform(df_orig[['sst']])

In [5]:
# ============================================================
# 3. CONSTRUCCIÓN DE REZAGOS EXCLUSIVAMENTE PARA TUS 6 VARIABLES
# ============================================================
# Definimos tus 6 variables seleccionadas como importantes
tus_6_variables = ['hum_esp_bc', 'prec_ln', 'dias_lluvia_ln', 'vel_vi_max_bc', 'temp_max_bc', 'sst_yj']
max_lag = 20

lags_dict = {}
for lag in range(1, max_lag + 1):
    for var in tus_6_variables:
        lags_dict[f'{var}_lag_{lag}'] = df_transf[var].shift(lag)

df_lags = pd.DataFrame(lags_dict, index=df_transf.index)
df_total = pd.concat([df_transf[['año', 'semana_epi', 'casos_ln']], df_lags], axis=1).dropna()

In [6]:
# ============================================================
# 4. DIVISIÓN CRONOLÓGICA (2021-2024 vs 1S-2025)
# ============================================================
df_train_cron = df_total[df_total['año'].isin([2021, 2022, 2023, 2024])]
df_test_cron = df_total[(df_total['año'] == 2025) & (df_total['semana_epi'] <= 26)]

variables_lags_generados = list(lags_dict.keys())
X_train = df_train_cron[variables_lags_generados]
X_test = df_test_cron[variables_lags_generados]

y_train = df_train_cron['casos_ln']
y_test = df_test_cron['casos_ln']

y_train_real = df_orig.loc[df_train_cron.index, 'casos_dengue']
y_test_real = df_orig.loc[df_test_cron.index, 'casos_dengue']

In [7]:
# ============================================================
# 5. ESTANDARIZACIÓN Y FILTRADO CON LASSO (DE 120 PASAMOS A LAS MEJORES)
# ============================================================
scaler_X = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train), columns=variables_lags_generados, index=y_train.index)
X_test_scaled = pd.DataFrame(scaler_X.transform(X_test), columns=variables_lags_generados, index=y_test.index)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

# Ejecución de LASSO CV
lasso_model = LassoCV(cv=5, random_state=42).fit(X_train_scaled, y_train_scaled)

# Extraer rezagos ganadores
coeficientes = pd.Series(lasso_model.coef_, index=variables_lags_generados)
vars_ganadoras_lasso = coeficientes[coeficientes != 0].index.tolist()

print("\n" + "="*80)
print(f"[LOG] De los 120 rezagos evaluados, LASSO seleccionó {len(vars_ganadoras_lasso)} variables:")
print("="*80)
for v in vars_ganadoras_lasso:
    print(f" -> {v} (Coeficiente: {coeficientes[v]:.4f})")
print("="*80)

# Si LASSO llegara a apagar todo por alta penalización, dejamos los 3 más fuertes por seguridad
if len(vars_ganadoras_lasso) == 0:
    vars_ganadoras_lasso = coeficientes.abs().nlargest(3).index.tolist()

X_train_lasso_sub = X_train_scaled[vars_ganadoras_lasso]
X_test_lasso_sub = X_test_scaled[vars_ganadoras_lasso]


[LOG] De los 120 rezagos evaluados, LASSO seleccionó 24 variables:
 -> hum_esp_bc_lag_1 (Coeficiente: 0.0812)
 -> prec_ln_lag_1 (Coeficiente: 0.0209)
 -> vel_vi_max_bc_lag_1 (Coeficiente: 0.0903)
 -> vel_vi_max_bc_lag_2 (Coeficiente: 0.0485)
 -> prec_ln_lag_3 (Coeficiente: 0.0412)
 -> vel_vi_max_bc_lag_3 (Coeficiente: 0.0074)
 -> prec_ln_lag_4 (Coeficiente: 0.0065)
 -> vel_vi_max_bc_lag_4 (Coeficiente: 0.0190)
 -> prec_ln_lag_5 (Coeficiente: 0.0373)
 -> vel_vi_max_bc_lag_5 (Coeficiente: 0.0676)
 -> prec_ln_lag_8 (Coeficiente: 0.0507)
 -> prec_ln_lag_9 (Coeficiente: 0.0360)
 -> prec_ln_lag_10 (Coeficiente: 0.0160)
 -> prec_ln_lag_11 (Coeficiente: 0.0030)
 -> dias_lluvia_ln_lag_11 (Coeficiente: 0.0208)
 -> dias_lluvia_ln_lag_14 (Coeficiente: 0.0379)
 -> vel_vi_max_bc_lag_15 (Coeficiente: 0.0290)
 -> vel_vi_max_bc_lag_16 (Coeficiente: 0.0064)
 -> vel_vi_max_bc_lag_17 (Coeficiente: 0.0286)
 -> dias_lluvia_ln_lag_19 (Coeficiente: 0.0175)
 -> sst_yj_lag_19 (Coeficiente: 0.1559)
 -> hum_esp_

In [8]:
# ============================================================
# 6. CONFIGURACIÓN COMPLETA DE ÓRDENES (CORREGIDA)
# ============================================================
arima_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1), (1,1,2), (2,1,2)]
arimax_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1), (1,1,2), (2,1,2)]

# Listas estacionales corregidas con las 5 configuraciones completas
sarima_orders = [
    ((1,1,1),(1,1,1,52)),
    ((1,1,0),(1,1,1,52)),
    ((0,1,1),(1,1,1,52)),
    ((2,1,1),(1,1,1,52)),
    ((1,1,2),(1,1,1,52))
]

sarimax_orders = [
    ((1,1,1),(1,1,1,52)),
    ((1,1,0),(1,1,1,52)),
    ((0,1,1),(1,1,1,52)),
    ((2,1,1),(1,1,1,52)),
    ((1,1,2),(1,1,1,52))
]

todas_las_metricas = []

def inversa_total_local(y_scaled_val):
    y_ln_val = scaler_y.inverse_transform(np.array(y_scaled_val).reshape(-1, 1)).flatten()
    return np.expm1(y_ln_val)

# Evaluación LASSO Puro
pred_lasso_tr = inversa_total_local(lasso_model.predict(X_train_scaled))
pred_lasso_te = inversa_total_local(lasso_model.predict(X_test_scaled))

todas_las_metricas.append({
    'Modelo': 'LASSO_Puro', 'Configuración': f'alpha={lasso_model.alpha_:.4f}',
    'MAE_Train': mean_absolute_error(y_train_real, pred_lasso_tr), 'MAE_Test': mean_absolute_error(y_test_real, pred_lasso_te),
    'RMSE_Train': np.sqrt(mean_squared_error(y_train_real, pred_lasso_tr)), 'RMSE_Test': np.sqrt(mean_squared_error(y_test_real, pred_lasso_te)),
    'R2_Train': r2_score(y_train_scaled, lasso_model.predict(X_train_scaled)), 'R2_Test': r2_score(y_test_scaled, lasso_model.predict(X_test_scaled))
})

# Ajustar y evaluar los experimentos Box-Jenkins
experimentos = [
    {'tipo': 'ARIMA', 'ordenes': [(o, (0,0,0,0)) for o in arima_orders], 'exog_tr': None, 'exog_te': None},
    {'tipo': 'SARIMA', 'ordenes': sarima_orders, 'exog_tr': None, 'exog_te': None},
    {'tipo': 'ARIMAX', 'ordenes': [(o, (0,0,0,0)) for o in arimax_orders], 'exog_tr': X_train_lasso_sub, 'exog_te': X_test_lasso_sub},
    {'tipo': 'SARIMAX', 'ordenes': sarimax_orders, 'exog_tr': X_train_lasso_sub, 'exog_te': X_test_lasso_sub}
]

for exp in experimentos:
    for orden_p, orden_s in exp['ordenes']:
        try:
            model = SARIMAX(endog=y_train, exog=exp['exog_tr'], order=orden_p, seasonal_order=orden_s)
            fitted_model = model.fit(disp=False)
            
            pred_tr_real = np.expm1(fitted_model.fittedvalues)
            pred_te_real = np.expm1(fitted_model.predict(start=y_test.index[0], end=y_test.index[-1], exog=exp['exog_te']))
            
            mae_tr = mean_absolute_error(y_train_real.iloc[1:], pred_tr_real.iloc[1:])
            mae_te = mean_absolute_error(y_test_real, pred_te_real)
            rmse_tr = np.sqrt(mean_squared_error(y_train_real.iloc[1:], pred_tr_real.iloc[1:]))
            rmse_te = np.sqrt(mean_squared_error(y_test_real, pred_te_real))
            r2_tr = r2_score(y_train.iloc[1:], fitted_model.fittedvalues.iloc[1:])
            r2_te = r2_score(y_test, fitted_model.predict(start=y_test.index[0], end=y_test.index[-1], exog=exp['exog_te']))
            
            str_orden = f"order={orden_p}" if exp['tipo'] in ['ARIMA','ARIMAX'] else f"order={orden_p} seasonal={orden_s}"
            todas_las_metricas.append({
                'Modelo': exp['tipo'], 'Configuración': str_orden,
                'MAE_Train': mae_tr, 'MAE_Test': mae_te, 'RMSE_Train': rmse_tr, 'RMSE_Test': rmse_te,
                'R2_Train': r2_tr, 'R2_Test': r2_te
            })
        except:
            continue

In [9]:
# ============================================================
# 7. TABLA FINAL RESUMIDA PARA LA TESIS
# ============================================================
df_res = pd.DataFrame(todas_las_metricas)
mejores_por_familia = df_res.loc[df_res.groupby('Modelo')['MAE_Test'].idxmin()]

print("\n" + "="*95 + "\n TABLA DE RESULTADOS DE TESIS COMPLETA (ÓRDENES ESTACIONALES CORREGIDOS) \n" + "="*95)
print(mejores_por_familia.to_string(index=False))


 TABLA DE RESULTADOS DE TESIS COMPLETA (ÓRDENES ESTACIONALES CORREGIDOS) 
    Modelo                          Configuración  MAE_Train  MAE_Test  RMSE_Train  RMSE_Test  R2_Train    R2_Test
     ARIMA                        order=(2, 1, 1)   5.028757 15.473581    7.707939  18.799823  0.822080  -0.012407
    ARIMAX                        order=(1, 1, 2)   4.883391 15.455358    7.391419  20.061458  0.853784  -0.390679
LASSO_Puro                           alpha=0.0548   8.220990 54.092829   13.682171  56.448894  0.682158 -32.743828
    SARIMA order=(1, 1, 0) seasonal=(1, 1, 1, 52)   7.867464 18.036793   12.408549  24.095009  0.636681  -0.668514
   SARIMAX order=(1, 1, 2) seasonal=(1, 1, 1, 52)   6.821829 21.052760    9.885512  27.462218  0.727862  -1.307626
